
# Reconstrucción de plots del grid search

Esta notebook reconstruye **solo** los gráficos del grid search que comparan arquitecturas:

1. Variando `p`.
2. Variando `lambda` / `lambd` para `BoostDropoutNet`, usando las demás arquitecturas como referencia constante.

Lee los resultados persistidos desde un `GRIDSEARCH_DIR` ya existente, **sin reentrenar** y **sin sobrescribir** plots originales.

La salida se guarda en:

```text
<GRIDSEARCH_DIR>/_reconstructions/<RECONSTRUCTION_NAME>_<timestamp>/
```


In [1]:

# =========================
# IMPORTS
# =========================
from pathlib import Path
import copy
import datetime as dt
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [12]:

# =========================
# CONFIGURACION
# =========================
# Reemplaza este path por el de tu sesion de grid search.
GRIDSEARCH_DIR = Path(r'runs/gridsearch/2026-04-16-22-02-26-910900-aec94d85')

# Si es None, usa todos los artifact types detectados / esperados.
SELECTED_ARTIFACT_TYPES = None

RECONSTRUCTION_NAME = 'gridsearch_replot'

DEFAULT_PLOT_CONFIG = {
    'dpi': 320,

    # Figuras combinadas: loss, accuracy y error en una fila.
    'combined_figsize': (24, 6.8),

    # Figuras individuales.
    'single_figsize': (8.2, 6.2),

    # Fuentes.
    'suptitle_fontsize': 24,
    'title_fontsize': 20,
    'label_fontsize': 21,
    'tick_fontsize': 19,
    'legend_fontsize': 19,

    # Estilo.
    'line_width': 2.4,
    'baseline_line_width': 2.6,
    'marker_size': 90,
    'grid_alpha': 0.35,

    # Control global de titulos.
    'show_axis_titles': False,
    'show_figure_suptitle': False,

    # Limites opcionales por metrica para plots variando p.
    # Claves soportadas: val_loss, val_acc, val_error, test_loss, test_acc, test_error.
    'p_y_limits': {},

    # Limites opcionales por metrica para plots variando lambda.
    # Claves soportadas: val_loss, val_acc, val_error, test_loss, test_acc, test_error.
    'lambda_y_limits': {},

    # Limites opcionales del eje x.
    # Ejemplos: 'p_x_limits': (0.0, 0.8), 'lambda_x_limits': (0.0, 0.1)
    'p_x_limits': None,
    'lambda_x_limits': None,
}

# Overrides recomendados para tesis/paper.
PLOT_CONFIG_OVERRIDES = {
    # 'show_axis_titles': False,
    # 'show_figure_suptitle': False,
    # 'p_y_limits': {
    #     'val_error': (0.0, 0.15),
    #     'val_loss': (0.0, 0.8),
    # },
    # 'lambda_y_limits': {
    #     'val_error': (0.0, 0.15),
    #     'val_loss': (0.0, 0.8),
    # },
}


In [5]:

# =========================
# CONSTANTES Y HELPERS
# =========================
GRIDSEARCH_ARTIFACT_TYPES = ['best_model', 'early_stopping', 'last_model']

MODEL_DISPLAY_NAMES = {
    'OverfitNet': 'OverfitNet',
    'DropoutNet': 'Dropout',
    'DropConnectNet': 'DropConnect',
    'BoostDropoutNet': 'BoostDropout',
}

MODEL_MARKERS = {
    'OverfitNet': 'o',
    'DropoutNet': 's',
    'DropConnectNet': 'D',
    'BoostDropoutNet': '^',
}

PAPER_MODEL_PALETTE = {
    'OverfitNet': '#4C78A8',
    'DropoutNet': '#F58518',
    'DropConnectNet': '#54A24B',
    'BoostDropoutNet': '#E45756',
}


def merge_plot_config(custom_config=None):
    cfg = copy.deepcopy(DEFAULT_PLOT_CONFIG)
    if custom_config:
        for key, value in custom_config.items():
            if isinstance(value, dict) and isinstance(cfg.get(key), dict):
                cfg[key].update(value)
            else:
                cfg[key] = value
    return cfg


def read_json(path):
    path = Path(path)
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def write_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def ensure_unique_directory(path):
    path = Path(path)
    if not path.exists():
        path.mkdir(parents=True, exist_ok=False)
        return path

    parent = path.parent
    stem = path.name
    idx = 1
    while True:
        candidate = parent / f'{stem}_{idx:02d}'
        if not candidate.exists():
            candidate.mkdir(parents=True, exist_ok=False)
            return candidate
        idx += 1


def make_reconstruction_root(gridsearch_dir, reconstruction_name='gridsearch_replot'):
    gridsearch_dir = Path(gridsearch_dir)
    timestamp = dt.datetime.now().strftime('%Y-%m-%d-%H-%M-%S-%f')
    base_dir = gridsearch_dir / '_reconstructions'
    base_dir.mkdir(parents=True, exist_ok=True)
    return ensure_unique_directory(base_dir / f'{reconstruction_name}_{timestamp}')


def _normalize_param_value(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, list):
        return tuple(_normalize_param_value(v) for v in value)
    if isinstance(value, dict):
        return tuple(sorted((k, _normalize_param_value(v)) for k, v in value.items()))
    return value


def _model_plot_color(model_name):
    return PAPER_MODEL_PALETTE.get(model_name, '#4C78A8')


def _load_model_grid_metadata(model_gridsearch_dir):
    return read_json(Path(model_gridsearch_dir) / 'model_gridsearch_metadata.json')


def _frozen_params_without_p(params):
    ignored = {'p', 'model_name'}
    return tuple(sorted(
        (k, _normalize_param_value(v))
        for k, v in params.items()
        if k not in ignored
    ))


def _frozen_params_without_keys(params, ignored_keys=None):
    ignored_keys = set(ignored_keys or [])
    return tuple(sorted(
        (k, _normalize_param_value(v))
        for k, v in params.items()
        if k not in ignored_keys
    ))


def maybe_set_axis_title(ax, title, cfg, **kwargs):
    if cfg.get('show_axis_titles', True) and title:
        ax.set_title(title, **kwargs)
    else:
        ax.set_title('')


def maybe_set_figure_suptitle(fig, title, cfg, **kwargs):
    if cfg.get('show_figure_suptitle', True) and title:
        fig.suptitle(title, **kwargs)


def _set_tick_fontsizes(ax, cfg):
    ax.tick_params(axis='both', labelsize=cfg['tick_fontsize'])


def _grid_metric_title_es(metric_key, split_name):
    split_name_es = {'val': 'validación', 'test': 'test'}.get(split_name, split_name)
    metric_name_es = {'loss': 'pérdida', 'acc': 'exactitud', 'error': 'error'}.get(metric_key, metric_key)
    return f'{metric_name_es.capitalize()} media en {split_name_es}'


def _grid_metric_ylabel_es(metric_key):
    return {'loss': 'Pérdida media', 'acc': 'Exactitud media', 'error': 'Error medio'}.get(metric_key, metric_key)


def _grid_metric_slug(metric_key, split_name):
    metric_slug = {'loss': 'perdida', 'acc': 'exactitud', 'error': 'error'}.get(metric_key, metric_key)
    split_slug = {'val': 'validacion', 'test': 'test'}.get(split_name, split_name)
    return f'{metric_slug}_{split_slug}'


def _lambda_metric_title_es(metric_key, split_name):
    return _grid_metric_title_es(metric_key, split_name)


def _lambda_metric_ylabel_es(metric_key):
    return _grid_metric_ylabel_es(metric_key)


def _lambda_metric_slug(metric_key, split_name):
    return _grid_metric_slug(metric_key, split_name)


In [6]:

# =========================
# CARGA Y SELECCION DE SERIES
# =========================

def resolve_gridsearch_metadata(gridsearch_dir):
    gridsearch_dir = Path(gridsearch_dir)
    metadata_path = gridsearch_dir / 'gridsearch_metadata.json'
    if not metadata_path.exists():
        raise FileNotFoundError(f'No existe gridsearch_metadata.json en: {gridsearch_dir}')

    metadata = read_json(metadata_path)

    # Repara model_gridsearch_dir si el gridsearch fue movido.
    for model_name, model_info in metadata.get('models', {}).items():
        current = Path(model_info.get('model_gridsearch_dir', ''))
        fallback = gridsearch_dir / model_name
        if current.exists():
            model_info['model_gridsearch_dir'] = str(current.resolve())
        elif fallback.exists():
            model_info['model_gridsearch_dir'] = str(fallback.resolve())

    return metadata


def detect_artifact_types(gridsearch_metadata):
    detected = set()
    for _, model_info in gridsearch_metadata.get('models', {}).items():
        model_grid_dir = Path(model_info['model_gridsearch_dir'])
        model_meta_path = model_grid_dir / 'model_gridsearch_metadata.json'
        if not model_meta_path.exists():
            continue
        model_meta = read_json(model_meta_path)
        for agg_run in model_meta.get('aggregated_runs', []):
            detected.update(agg_run.get('artifact_aggregates', {}).keys())

    if detected:
        ordered = [a for a in GRIDSEARCH_ARTIFACT_TYPES if a in detected]
        ordered += sorted(a for a in detected if a not in ordered)
        return ordered
    return GRIDSEARCH_ARTIFACT_TYPES


def _aggregate_record_for_plot(agg_run, artifact_type):
    artifact_data = agg_run.get('artifact_aggregates', {}).get(artifact_type)
    if not artifact_data or artifact_data.get('mean_val_loss') is None:
        return None

    params = dict(agg_run.get('group_params', {}))
    model_name = agg_run.get('model_name')

    mean_val_acc = artifact_data.get('mean_val_acc')
    mean_test_acc = artifact_data.get('mean_test_acc')
    std_val_acc = artifact_data.get('std_val_acc')
    std_test_acc = artifact_data.get('std_test_acc')

    return {
        'model_name': model_name,
        'artifact_type': artifact_type,
        'combo_id': agg_run.get('combo_id'),
        'p': params.get('p'),
        'params': params,
        'comparison_key': _frozen_params_without_p(params),
        'val_loss': artifact_data.get('mean_val_loss'),
        'val_acc': mean_val_acc,
        'val_error': None if mean_val_acc is None else float(1.0 - mean_val_acc),
        'test_loss': artifact_data.get('mean_test_loss'),
        'test_acc': mean_test_acc,
        'test_error': None if mean_test_acc is None else float(1.0 - mean_test_acc),
        'std_val_loss': artifact_data.get('std_val_loss'),
        'std_val_acc': std_val_acc,
        'std_val_error': std_val_acc,
        'std_test_loss': artifact_data.get('std_test_loss'),
        'std_test_acc': std_test_acc,
        'std_test_error': std_test_acc,
        'num_seed_runs': artifact_data.get('num_seed_runs'),
    }


def _select_reference_group(records):
    valid_records = [r for r in records if r['val_loss'] is not None]
    if not valid_records:
        return None, []

    best_record = min(valid_records, key=lambda r: r['val_loss'])
    key = best_record['comparison_key']
    matched = [r for r in valid_records if r['comparison_key'] == key]
    matched = sorted(matched, key=lambda r: (-1 if r['p'] is None else float(r['p'])))
    return best_record, matched


def _expand_baseline_points(series, x_reference):
    if not series or not x_reference:
        return series
    if all(item['p'] is None for item in series):
        template = dict(series[0])
        expanded = []
        for p in x_reference:
            new_item = dict(template)
            new_item['p'] = p
            expanded.append(new_item)
        return expanded
    return series


def build_series_varying_p(gridsearch_metadata, artifact_type):
    model_runs = {}
    selection_summary = {}

    for model_name, model_info in gridsearch_metadata.get('models', {}).items():
        model_grid_dir = Path(model_info['model_gridsearch_dir'])
        model_meta = _load_model_grid_metadata(model_grid_dir)
        aggregated_runs = model_meta.get('aggregated_runs', [])

        records = []
        for agg_run in aggregated_runs:
            record = _aggregate_record_for_plot(agg_run, artifact_type)
            if record is not None:
                records.append(record)

        best_record, matched_records = _select_reference_group(records)
        model_runs[model_name] = matched_records

        if best_record is not None:
            selection_summary[model_name] = {
                'reference_combo_id': best_record['combo_id'],
                'reference_params_fixed_except_p': {
                    k: v for k, v in best_record['params'].items()
                    if k not in {'p', 'model_name'}
                },
                'matched_combo_ids': [r['combo_id'] for r in matched_records],
                'matched_p_values': [r['p'] for r in matched_records],
            }

    union_p = sorted({
        float(item['p'])
        for series in model_runs.values()
        for item in series
        if item.get('p') is not None
    })

    expanded = {
        model_name: _expand_baseline_points(series, union_p)
        for model_name, series in model_runs.items()
    }

    return expanded, selection_summary


def _build_lambda_reference_record(agg_run, artifact_type):
    record = _aggregate_record_for_plot(agg_run, artifact_type)
    if record is None:
        return None
    record['lambd'] = record['params'].get('lambd')
    return record


def _select_reference_group_varying_lambda(records):
    if not records:
        return None, []

    valid_records = [r for r in records if r.get('val_loss') is not None]
    if not valid_records:
        return None, []

    best_record = min(valid_records, key=lambda r: (r['val_loss'], -(r.get('val_acc') or float('-inf'))))
    best_params = best_record['params']

    if best_record['model_name'].lower() == 'boostdropoutnet':
        ref_key = _frozen_params_without_keys(best_params, ignored_keys={'lambd', 'model_name'})
        matched = [
            r for r in records
            if r['model_name'].lower() == 'boostdropoutnet'
            and _frozen_params_without_keys(r['params'], ignored_keys={'lambd', 'model_name'}) == ref_key
        ]
        matched = sorted(
            matched,
            key=lambda r: float('-inf') if r['params'].get('lambd') is None else r['params'].get('lambd')
        )
        return best_record, matched

    return best_record, [best_record]


def build_series_varying_lambda(gridsearch_metadata, artifact_type):
    per_model = {}
    selection_summary = {}

    for model_name, model_info in gridsearch_metadata.get('models', {}).items():
        model_grid_dir = Path(model_info['model_gridsearch_dir'])
        model_meta = _load_model_grid_metadata(model_grid_dir)
        aggregated_runs = model_meta.get('aggregated_runs', [])

        records = []
        for agg_run in aggregated_runs:
            record = _build_lambda_reference_record(agg_run, artifact_type)
            if record is not None:
                records.append(record)

        best_record, matched_records = _select_reference_group_varying_lambda(records)
        if matched_records:
            per_model[model_name] = matched_records

        if best_record is not None:
            selection_summary[model_name] = {
                'reference_combo_id': best_record['combo_id'],
                'reference_params': dict(best_record['params']),
                'matched_combo_ids': [r['combo_id'] for r in matched_records],
                'matched_lambda_values': [r.get('lambd') for r in matched_records],
            }

    boost_series = per_model.get('BoostDropoutNet', [])
    lambda_values = sorted({
        float(item['lambd'])
        for item in boost_series
        if item.get('lambd') is not None
    })

    if not lambda_values:
        return {}, selection_summary

    expanded = {}
    for model_name, series in per_model.items():
        if model_name.lower() == 'boostdropoutnet':
            expanded[model_name] = series
        else:
            if not series:
                continue
            reference_item = dict(series[0])
            clones = []
            for lambd in lambda_values:
                clone = dict(reference_item)
                clone['lambd'] = lambd
                clones.append(clone)
            expanded[model_name] = clones

    return expanded, selection_summary


In [14]:

# =========================
# PLOTTING
# =========================

def _apply_axis_limits(ax, metric_key, cfg, mode):
    if mode == 'p':
        y_limits = cfg.get('p_y_limits', {})
        x_limits = cfg.get('p_x_limits')
    elif mode == 'lambda':
        y_limits = cfg.get('lambda_y_limits', {})
        x_limits = cfg.get('lambda_x_limits')
    else:
        y_limits = {}
        x_limits = None

    if y_limits.get(metric_key) is not None:
        ax.set_ylim(*y_limits[metric_key])
    if x_limits is not None:
        ax.set_xlim(*x_limits)


def _plot_metric_axis(ax, series_by_model, metric_key, title, ylabel=None, cfg=None, mode='p'):
    cfg = merge_plot_config(cfg)
    has_data = False

    for model_name, series in series_by_model.items():
        valid_series = [item for item in series if item.get(metric_key) is not None and item.get('p') is not None]
        if not valid_series:
            continue

        has_data = True
        valid_series = sorted(valid_series, key=lambda item: float(item['p']))
        xs = [float(item['p']) for item in valid_series]
        ys = [item[metric_key] for item in valid_series]
        color = _model_plot_color(model_name)

        if model_name.lower() == 'overfitnet':
            ax.plot(
                xs, ys,
                linewidth=cfg['baseline_line_width'],
                alpha=0.75,
                linestyle='--',
                color=color,
                label=f"{MODEL_DISPLAY_NAMES.get(model_name, model_name)}",
                zorder=2,
            )
        else:
            ax.plot(
                xs, ys,
                linewidth=cfg['line_width'],
                alpha=0.9,
                marker=MODEL_MARKERS.get(model_name, 'o'),
                color=color,
                label=MODEL_DISPLAY_NAMES.get(model_name, model_name),
                zorder=3,
            )
            ax.scatter(
                xs, ys,
                s=cfg['marker_size'],
                marker=MODEL_MARKERS.get(model_name, 'o'),
                color=color,
                alpha=0.9,
                zorder=4,
            )

    maybe_set_axis_title(ax, title, cfg, fontsize=cfg['title_fontsize'])
    ax.set_xlabel('p', fontsize=cfg['label_fontsize'])
    ax.set_ylabel(ylabel if ylabel is not None else metric_key.replace('_', ' ').title(), fontsize=cfg['label_fontsize'])
    ax.grid(True, linestyle='--', alpha=cfg['grid_alpha'])
    _set_tick_fontsizes(ax, cfg)
    _apply_axis_limits(ax, metric_key, cfg, mode='p')

    if has_data:
        ax.legend(fontsize=cfg['legend_fontsize'], loc='best', frameon=False)
    else:
        ax.text(0.5, 0.5, 'Sin datos disponibles', ha='center', va='center', transform=ax.transAxes)


def _plot_metric_axis_vs_lambda(ax, series_by_model, metric_key, title, ylabel=None, cfg=None):
    cfg = merge_plot_config(cfg)
    has_data = False

    for model_name, series in series_by_model.items():
        valid_series = [item for item in series if item.get(metric_key) is not None and item.get('lambd') is not None]
        if not valid_series:
            continue

        has_data = True
        valid_series = sorted(valid_series, key=lambda item: float(item['lambd']))
        xs = [float(item['lambd']) for item in valid_series]
        ys = [item[metric_key] for item in valid_series]
        color = _model_plot_color(model_name)

        if model_name.lower() in {'overfitnet', 'dropoutnet', 'dropconnectnet'}:
            ax.plot(
                xs, ys,
                linewidth=cfg['baseline_line_width'],
                alpha=0.75,
                linestyle='--',
                color=color,
                label=f"{MODEL_DISPLAY_NAMES.get(model_name, model_name)}",
                zorder=2,
            )
        else:
            ax.plot(
                xs, ys,
                linewidth=cfg['line_width'],
                alpha=0.9,
                marker=MODEL_MARKERS.get(model_name, 'o'),
                color=color,
                label=MODEL_DISPLAY_NAMES.get(model_name, model_name),
                zorder=3,
            )
            ax.scatter(
                xs, ys,
                s=cfg['marker_size'],
                marker=MODEL_MARKERS.get(model_name, 'o'),
                color=color,
                alpha=0.9,
                zorder=4,
            )

    maybe_set_axis_title(ax, title, cfg, fontsize=cfg['title_fontsize'])
    ax.set_xlabel('λ', fontsize=cfg['label_fontsize'])
    ax.set_ylabel(ylabel if ylabel is not None else metric_key.replace('_', ' ').title(), fontsize=cfg['label_fontsize'])
    ax.grid(True, linestyle='--', alpha=cfg['grid_alpha'])
    _set_tick_fontsizes(ax, cfg)
    _apply_axis_limits(ax, metric_key, cfg, mode='lambda')

    if has_data:
        ax.legend(fontsize=cfg['legend_fontsize'], loc='best', frameon=False)
    else:
        ax.text(0.5, 0.5, 'Sin datos disponibles', ha='center', va='center', transform=ax.transAxes)


def plot_gridsearch_varying_p(series_by_model, artifact_type, output_dir, plot_config=None):
    cfg = merge_plot_config(plot_config)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    outputs = {}

    split_specs = [
        (
            "val",
            [
                ("val_loss", "loss", "val"),
                ("val_acc", "acc", "val"),
                ("val_error", "error", "val"),
            ],
            "validacion",
            "Comparación de grid search variando p en validación",
        ),
        (
            "test",
            [
                ("test_loss", "loss", "test"),
                ("test_acc", "acc", "test"),
                ("test_error", "error", "test"),
            ],
            "test",
            "Comparación de grid search variando p en test",
        ),
    ]

    for split_key, metric_specs, split_slug, suptitle in split_specs:
        fig, axes = plt.subplots(1, 3, figsize=cfg["combined_figsize"])

        for ax, (metric_field, metric_key, split_name) in zip(axes, metric_specs):
            _plot_metric_axis(
                ax,
                series_by_model,
                metric_key=metric_field,
                title=_grid_metric_title_es(metric_key, split_name),
                ylabel=_grid_metric_ylabel_es(metric_key),
                cfg=cfg,
                mode="p",
            )

        maybe_set_figure_suptitle(
            fig,
            suptitle,
            cfg,
            fontsize=cfg["suptitle_fontsize"],
        )

        fig.tight_layout(
            rect=[0, 0, 1, 0.96] if cfg.get("show_figure_suptitle", True) else None
        )

        combined_path = output_dir / f"comparison_p_{split_slug}_{artifact_type}.png"
        fig.savefig(combined_path, dpi=cfg["dpi"], bbox_inches="tight")
        plt.close(fig)

        individual_paths = []

        for metric_field, metric_key, split_name in metric_specs:
            fig_single, ax_single = plt.subplots(figsize=cfg["single_figsize"])

            _plot_metric_axis(
                ax_single,
                series_by_model,
                metric_key=metric_field,
                title=_grid_metric_title_es(metric_key, split_name),
                ylabel=_grid_metric_ylabel_es(metric_key),
                cfg=cfg,
                mode="p",
            )

            fig_single.tight_layout()

            single_path = output_dir / (
                f"comparison_p_{_grid_metric_slug(metric_key, split_name)}_{artifact_type}.png"
            )

            fig_single.savefig(single_path, dpi=cfg["dpi"], bbox_inches="tight")
            plt.close(fig_single)

            individual_paths.append(str(single_path))

        outputs[split_key] = {
            "combined_plot": str(combined_path),
            "individual_plots": individual_paths,
        }

    return outputs


def plot_gridsearch_varying_lambda(series_by_model, artifact_type, output_dir, plot_config=None):
    cfg = merge_plot_config(plot_config)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    outputs = {}

    for split_name, metric_specs, combined_name, suptitle in [
        ('val', [('val_loss', 'loss', 'val'), ('val_acc', 'acc', 'val'), ('val_error', 'error', 'val')], 'validation', 'Comparación de grid search variando λ en BoostDropout'),
        ('test', [('test_loss', 'loss', 'test'), ('test_acc', 'acc', 'test'), ('test_error', 'error', 'test')], 'test', 'Comparación en test variando λ en BoostDropout'),
    ]:
        fig, axes = plt.subplots(1, 3, figsize=cfg['combined_figsize'])
        for ax, (metric_field, metric_key, split) in zip(axes, metric_specs):
            _plot_metric_axis_vs_lambda(
                ax,
                series_by_model,
                metric_key=metric_field,
                title=_lambda_metric_title_es(metric_key, split),
                ylabel=_lambda_metric_ylabel_es(metric_key),
                cfg=cfg,
            )

        maybe_set_figure_suptitle(fig, suptitle, cfg, fontsize=cfg['suptitle_fontsize'])
        fig.tight_layout(rect=[0, 0, 1, 0.96] if cfg.get('show_figure_suptitle', True) else None)

        combined_path = output_dir / f'comparison_lambda_{combined_name}_{artifact_type}.png'
        fig.savefig(combined_path, dpi=cfg['dpi'], bbox_inches='tight')
        plt.close(fig)

        individual_paths = []
        for metric_field, metric_key, split in metric_specs:
            fig_single, ax_single = plt.subplots(figsize=cfg['single_figsize'])
            _plot_metric_axis_vs_lambda(
                ax_single,
                series_by_model,
                metric_key=metric_field,
                title=_lambda_metric_title_es(metric_key, split),
                ylabel=_lambda_metric_ylabel_es(metric_key),
                cfg=cfg,
            )
            fig_single.tight_layout()
            single_path = output_dir / f'comparison_lambda_{_lambda_metric_slug(metric_key, split)}_{artifact_type}.png'
            fig_single.savefig(single_path, dpi=cfg['dpi'], bbox_inches='tight')
            plt.close(fig_single)
            individual_paths.append(str(single_path))

        outputs[split_name] = {
            'combined_plot': str(combined_path),
            'individual_plots': individual_paths,
        }

    return outputs


In [8]:

# =========================
# RECONSTRUCCION COMPLETA
# =========================

def reconstruct_gridsearch_comparison_plots(
    gridsearch_dir,
    selected_artifact_types=None,
    reconstruction_name='gridsearch_replot',
    plot_config=None,
    rebuild_p_plots=True,
    rebuild_lambda_plots=True,
):
    gridsearch_dir = Path(gridsearch_dir)
    if not gridsearch_dir.exists():
        raise FileNotFoundError(f'No existe GRIDSEARCH_DIR: {gridsearch_dir}')

    cfg = merge_plot_config(plot_config)
    gridsearch_metadata = resolve_gridsearch_metadata(gridsearch_dir)

    if selected_artifact_types is None:
        selected_artifact_types = detect_artifact_types(gridsearch_metadata)

    reconstruction_root = make_reconstruction_root(gridsearch_dir, reconstruction_name=reconstruction_name)

    result = {
        'source_gridsearch_dir': str(gridsearch_dir.resolve()),
        'reconstruction_root': str(reconstruction_root.resolve()),
        'selected_artifact_types': list(selected_artifact_types),
        'plot_config': cfg,
        'generated_at': dt.datetime.now().isoformat(),
        'artifacts': {},
    }

    for artifact_type in selected_artifact_types:
        print('\n' + '=' * 100)
        print(f'🔁 Reconstruyendo grid search para artifact_type={artifact_type}')
        print('=' * 100)

        artifact_result = {}

        if rebuild_p_plots:
            p_dir = reconstruction_root / 'varying_p' / artifact_type
            series_p, selection_p = build_series_varying_p(gridsearch_metadata, artifact_type)
            if any(series for series in series_p.values()):
                artifact_result['varying_p'] = {
                    'selection': selection_p,
                    'plots': plot_gridsearch_varying_p(
                        series_by_model=series_p,
                        artifact_type=artifact_type,
                        output_dir=p_dir,
                        plot_config=cfg,
                    ),
                }
            else:
                print(f'⚠️ Sin datos para plots variando p: {artifact_type}')
                artifact_result['varying_p'] = {'selection': selection_p, 'plots': None}

        if rebuild_lambda_plots:
            lambda_dir = reconstruction_root / 'varying_lambda' / artifact_type
            series_lambda, selection_lambda = build_series_varying_lambda(gridsearch_metadata, artifact_type)
            if any(series for series in series_lambda.values()):
                artifact_result['varying_lambda'] = {
                    'selection': selection_lambda,
                    'plots': plot_gridsearch_varying_lambda(
                        series_by_model=series_lambda,
                        artifact_type=artifact_type,
                        output_dir=lambda_dir,
                        plot_config=cfg,
                    ),
                }
            else:
                print(f'⚠️ Sin datos para plots variando lambda: {artifact_type}')
                artifact_result['varying_lambda'] = {'selection': selection_lambda, 'plots': None}

        result['artifacts'][artifact_type] = artifact_result

    write_json(result, reconstruction_root / 'gridsearch_reconstruction_metadata.json')

    print('\n' + '=' * 100)
    print('✅ RECONSTRUCCION DE GRID SEARCH FINALIZADA')
    print('=' * 100)
    print(f'📁 Salida: {reconstruction_root}')
    print('⚠️ Los plots originales del grid search no fueron modificados.')

    return result


In [15]:

# =========================
# EJECUCION
# =========================
plot_config = merge_plot_config(PLOT_CONFIG_OVERRIDES)

# Descomenta cuando hayas configurado GRIDSEARCH_DIR.
results = reconstruct_gridsearch_comparison_plots(
    gridsearch_dir=GRIDSEARCH_DIR,
    selected_artifact_types=SELECTED_ARTIFACT_TYPES,
    reconstruction_name=RECONSTRUCTION_NAME,
    plot_config=plot_config,
    rebuild_p_plots=True,
    rebuild_lambda_plots=False,
)
# results



🔁 Reconstruyendo grid search para artifact_type=best_model

🔁 Reconstruyendo grid search para artifact_type=early_stopping

🔁 Reconstruyendo grid search para artifact_type=last_model

✅ RECONSTRUCCION DE GRID SEARCH FINALIZADA
📁 Salida: runs/gridsearch/2026-04-16-22-02-26-910900-aec94d85/_reconstructions/gridsearch_replot_2026-04-29-21-26-29-167937
⚠️ Los plots originales del grid search no fueron modificados.



## Ejemplos de overrides útiles

### Quitar todos los títulos

```python
PLOT_CONFIG_OVERRIDES = {
    'show_axis_titles': False,
    'show_figure_suptitle': False,
}
```

### Agrandar letras

```python
PLOT_CONFIG_OVERRIDES = {
    'label_fontsize': 18,
    'tick_fontsize': 16,
    'legend_fontsize': 15,
    'title_fontsize': 20,
    'suptitle_fontsize': 24,
}
```

### Fijar límites del eje Y

```python
PLOT_CONFIG_OVERRIDES = {
    'p_y_limits': {
        'val_error': (0.0, 0.15),
        'val_loss': (0.0, 0.8),
    },
    'lambda_y_limits': {
        'val_error': (0.0, 0.15),
        'val_loss': (0.0, 0.8),
        'test_error': (0.0, 0.15),
        'test_loss': (0.0, 0.8),
    },
}
```

### Fijar límites del eje X

```python
PLOT_CONFIG_OVERRIDES = {
    'p_x_limits': (0.0, 0.8),
    'lambda_x_limits': (0.0, 0.1),
}
```
